# Pika H3 Flex · 官方六步影音生成

指定 `pika` 使用此入口；`omni` 使用 [ComfyUI notebook](ComfyUI_Colab.ipynb) 的 `h3-omni-int8`。工具已內嵌，不需 clone colab-utils。依固定版本安裝官方 `pika-h3-pro`，保留專用採樣流程。

**預設不安裝、不下載權重、不生成。** 先核對 GPU／host RAM／磁碟，再明確執行。官方以 H100 80GB 測試，此接入尚未做 GPU 實測。兩種 task 家族合計約 210 GB base＋5.5 GB adapters，另留依賴與暫存空間；只在生成時下載所選模式。CPU offload 不卸載 encoder／VAEs，也需要大量 host RAM。安裝使用 Python 3.12 隔離 venv，需要 NVIDIA runtime、git、patch、FFmpeg 與 C compiler，不配置付費資源。

原生 480p／768p、24fps、5–15 整數秒（實際依 17n+5 對齊）、32kHz stereo。4K 與分離 audio stems 是另外的 hosted API，未接入。[工具說明與限制](../skills/minimax-h3-colab/references/pika-h3-flex.md)。

## A. 設定與安裝


In [ ]:
# @title 1. 還原工具與離線計畫（不下載權重）
import json
import sys
from pathlib import Path
WORKSPACE_ROOT = "/content/pika-h3" # @param {type:"string"}
TOOLKIT_ROOT = "/content/pika-h3-toolkit" # @param {type:"string"}
PIKA_SOURCES = {'pika_h3.py': '#!/usr/bin/env python3\n"""Offline planning and isolated Colab execution of the official Pika H3 Flex CLI.\n\nLocal imports require only Python 3.10+ stdlib. Install/generate are explicit;\nthis tool never allocates a Colab runtime or calls a hosted Pika API.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom datetime import datetime, timezone\nfrom fractions import Fraction\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport uuid\n\nLOCK = json.loads(Path(__file__).with_name("pika_h3_lock.json").read_text())\nTASKS = ("t2va", "i2va", "l2va", "fl2va", "ref2va")\n\n\ndef save_json(path, data):\n    path = Path(path)\n    pending = path.with_suffix(path.suffix + ".tmp")\n    pending.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\\n")\n    pending.replace(path)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef request(*, prompt, task="t2va", resolution="480p", aspect="16:9", duration=5,\n            seed=0, first_image=None, last_image=None, refs=(), cpu_offload=False):\n    """Validate routing without reading media, downloading or touching the GPU.\n\n    Source duration/format validation and reference hashes are supplied by the\n    official CLI when it loads the real inputs.\n    """\n    if task not in TASKS:\n        raise ValueError(f"Unknown Pika task: {task}")\n    if not isinstance(prompt, str) or not prompt.strip() or len(prompt) > 8000:\n        raise ValueError("Pika prompt must contain 1–8000 characters")\n    if resolution not in LOCK["resolution_buckets"] or aspect not in LOCK["resolution_buckets"][resolution]:\n        raise ValueError("Choose a supported Pika resolution/aspect bucket")\n    if type(duration) is not int or not 5 <= duration <= 15:\n        raise ValueError("Pika duration must be an integer from 5 to 15 seconds")\n    if type(seed) is not int or not 0 <= seed < 2**63:\n        raise ValueError("Pika seed must be an integer from 0 to 2**63 - 1")\n    if type(cpu_offload) is not bool or not isinstance(refs, (list, tuple)):\n        raise ValueError("cpu_offload must be boolean and refs must be an ordered list")\n    first_image = str(first_image) if first_image else None\n    last_image = str(last_image) if last_image else None\n    if bool(first_image) != (task in ("i2va", "fl2va")):\n        raise ValueError("first_image is required only for i2va/fl2va")\n    if bool(last_image) != (task in ("l2va", "fl2va")):\n        raise ValueError("last_image is required only for l2va/fl2va")\n    counts = {"image": 0, "video": 0, "audio": 0}\n    for ref in refs:\n        kind, separator, path = ref.partition(":") if isinstance(ref, str) else ("", "", "")\n        if kind not in counts or not separator or not path.strip():\n            raise ValueError("References must use image:path, video:path or audio:path")\n        counts[kind] += 1\n    if task == "ref2va":\n        if not counts["image"] + counts["video"]:\n            raise ValueError("ref2va requires at least one image or video reference")\n        if len(refs) > 12 or counts["image"] > 9 or counts["video"] > 3 or counts["audio"] > 3:\n            raise ValueError("Pika limits: 9 images, 3 videos, 3 audio clips, 12 references total")\n    elif refs:\n        raise ValueError("References require task=ref2va; endpoint images use first_image/last_image")\n    return dict(task=task, prompt=prompt, resolution=resolution, aspect=aspect, duration=duration,\n                seed=seed, first_image=first_image, last_image=last_image,\n                refs=list(refs), cpu_offload=cpu_offload)\n\n\ndef geometry(job):\n    width, height = LOCK["resolution_buckets"][job["resolution"]][job["aspect"]]\n    frames = ((job["duration"] * 24 - 5 + 16) // 17) * 17 + 5\n    return dict(width=width, height=height, fps=24, frames=frames, seconds=frames / 24)\n\n\nclass PikaWorkspace:\n    def __init__(self, root="/content/pika-h3"):\n        self.root = Path(root).expanduser().resolve()\n        self.source = self.root / "source"\n        self.python = self.root / "venv/bin/python"\n        self.executable = self.root / "venv/bin/pika-h3-pro"\n        self.marker = self.root / "install.json"\n\n    def plan(self):\n        """Pure offline plan; Hugging Face\'s existing cache/environment is reused."""\n        return dict(tool="pika", root=str(self.root), release=LOCK,\n                    install_python="3.12 (isolated venv)",\n                    tested_upstream_gpu="H100 80GB; local GPU validation not_run",\n                    disk_note="Both tasks together: about 210 GB base + 5.5 GB adapters, plus staging/dependencies",\n                    memory_note="CPU offload does not offload the text encoder or VAEs; check host RAM too",\n                    tasks=list(TASKS), model_download="Selected task downloads on explicit generate only")\n\n    def command(self, job, prompt_file, output):\n        command = [str(self.executable), "--task", job["task"], "--prompt-file", str(prompt_file),\n                   "--model-repo", LOCK["model_repo"], "--revision", LOCK["model_revision"],\n                   "--resolution", job["resolution"], "--aspect", job["aspect"],\n                   "--duration", str(job["duration"]), "--seed", str(job["seed"]), "--output", str(output)]\n        for flag, value in (("--first-image", job["first_image"]), ("--last-image", job["last_image"])):\n            if value:\n                command.extend((flag, value))\n        for ref in job["refs"]:\n            command.extend(("--ref", ref))\n        if job["cpu_offload"]:\n            command.append("--cpu-offload")\n        return command\n\n    def prepare(self, **kwargs):\n        job = request(**kwargs)\n        directory = self.root / "runs/<new-run-id>"\n        return dict(request=job, geometry=geometry(job),\n                    command=self.command(job, directory / "prompt.txt", directory / "video.mp4"))\n\n    def _installed(self):\n        if not self.marker.is_file() or not self.executable.is_file():\n            return False\n        return json.loads(self.marker.read_text()).get("release") == LOCK\n\n    def _verify_source(self):\n        head = subprocess.check_output(["git", "-C", str(self.source), "rev-parse", "HEAD"], text=True).strip()\n        dirty = subprocess.check_output(["git", "-C", str(self.source), "status", "--porcelain"], text=True).strip()\n        if head != LOCK["code_revision"] or dirty:\n            raise RuntimeError("Pika source differs from the pinned release; preserve it and use another workspace root")\n\n    def install(self, python=None):\n        if self._installed():\n            self._verify_source()\n            return dict(installed=True, reused=True, root=str(self.root))\n        python = python or (sys.executable if sys.version_info[:2] == (3, 12) else shutil.which("python3.12"))\n        if not python:\n            raise RuntimeError("Official Pika requires Python 3.12; supply --python /path/to/python3.12")\n        version = subprocess.check_output([str(python), "-c", "import sys; print(\'%d.%d\' % sys.version_info[:2])"], text=True).strip()\n        if version != LOCK["python"]:\n            raise RuntimeError("Official Pika requires Python 3.12; existing environments are not modified")\n        for program in ("git", "patch", "ffmpeg", "ffprobe", "cc", "nvidia-smi"):\n            if not shutil.which(program):\n                raise RuntimeError(f"Install prerequisite on the target GPU runtime: {program}")\n        self.root.mkdir(parents=True, exist_ok=True)\n        with (self.root / "install.log").open("a") as log:\n            def run(args, cwd=None):\n                subprocess.run(list(map(str, args)), cwd=cwd, stdout=log, stderr=subprocess.STDOUT, check=True)\n\n            if not self.source.exists():\n                run(["git", "clone", "--no-checkout", LOCK["code_repo"], self.source])\n                run(["git", "checkout", "--detach", LOCK["code_revision"]], cwd=self.source)\n            self._verify_source()\n            run([python, "-m", "venv", self.root / "venv"])\n            run([self.python, "-m", "pip", "install", "-r", self.source / "requirements.txt"])\n            run([self.python, self.source / "tools/apply_diffusers_patches.py"], cwd=self.source)\n            run([self.python, "-m", "pip", "install", "--no-deps", "-e", self.source])\n            run([self.executable, "--help"])\n        save_json(self.marker, dict(release=LOCK, installed_at=datetime.now(timezone.utc).isoformat()))\n        return dict(installed=True, reused=False, root=str(self.root), log=str(self.root / "install.log"))\n\n    def generate(self, **kwargs):\n        job = request(**kwargs)\n        for value in [job["first_image"], job["last_image"], *[ref.split(":", 1)[1] for ref in job["refs"]]]:\n            if value and not Path(value).is_file():\n                raise ValueError(f"Input file does not exist on this runtime: {value}")\n        if not self._installed():\n            raise RuntimeError("Run explicit install first; generate does not install or switch models")\n        self._verify_source()\n        directory = self.root / "runs" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:10])\n        directory.mkdir(parents=True, exist_ok=False)\n        prompt_file, output = directory / "prompt.txt", directory / "video.mp4"\n        prompt_file.write_text(job["prompt"], encoding="utf-8")\n        prepared = dict(request=job, geometry=geometry(job), release=LOCK,\n                        command=self.command(job, prompt_file, output))\n        save_json(directory / "request.json", prepared)\n        receipt = dict(run_id=directory.name, status="starting", directory=str(directory),\n                       started_at=datetime.now(timezone.utc).isoformat(), content_verdict="not_reviewed")\n        receipt_path = directory / "receipt.json"\n        save_json(receipt_path, receipt)\n        started = time.monotonic()\n        try:\n            with (directory / "generation.log").open("w") as log:\n                process = subprocess.Popen(prepared["command"], stdout=log, stderr=subprocess.STDOUT)\n                receipt.update(status="running", pid=process.pid)\n                save_json(receipt_path, receipt)\n                if process.wait():\n                    raise RuntimeError(f"Pika failed; inspect {directory / \'generation.log\'}")\n            metadata = output.with_suffix(".json")\n            json.loads(metadata.read_text())\n            probe = json.loads(subprocess.check_output([\n                "ffprobe", "-v", "error", "-count_frames", "-show_streams", "-of", "json", str(output)\n            ], text=True))\n            video = next(stream for stream in probe["streams"] if stream["codec_type"] == "video")\n            audio = next(stream for stream in probe["streams"] if stream["codec_type"] == "audio")\n            expected = prepared["geometry"]\n            if (video["width"], video["height"], int(video["nb_read_frames"])) != (expected["width"], expected["height"], expected["frames"]):\n                raise RuntimeError("Pika output geometry/frame count differs from the requested bucket")\n            if audio.get("channels") != 2 or int(audio.get("sample_rate", 0)) != 32000:\n                raise RuntimeError("Pika output must contain 32 kHz stereo audio")\n            if (Fraction(video["avg_frame_rate"]) != 24 or video["codec_name"] != "h264"\n                    or audio["codec_name"] != "aac"\n                    or abs(float(audio["duration"]) - expected["seconds"]) > 0.15):\n                raise RuntimeError("Pika output must be 24fps H.264/AAC with a full-length soundtrack")\n            subprocess.run(["ffmpeg", "-v", "error", "-xerror", "-i", str(output), "-f", "null", "-"],\n                           stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, check=True)\n            receipt.update(status="technically_valid", video=str(output), metadata=str(metadata),\n                           video_sha256=sha256(output), metadata_sha256=sha256(metadata), media=probe)\n        except (KeyboardInterrupt, SystemExit):\n            receipt["status"] = "interrupted_needs_reconcile"\n            raise\n        except Exception as error:\n            receipt.update(status="failed", error=str(error))\n            raise\n        finally:\n            receipt.update(wall_seconds=time.monotonic() - started,\n                           updated_at=datetime.now(timezone.utc).isoformat())\n            save_json(receipt_path, receipt)\n        return receipt\n\n    def status(self):\n        receipts = sorted((self.root / "runs").glob("*/receipt.json"))[-5:]\n        return dict(root=str(self.root), installed=self._installed(),\n                    recent_runs=[json.loads(path.read_text()) for path in receipts])\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest="action", required=True)\n    for action in ("plan", "install", "generate", "status"):\n        sub = commands.add_parser(action)\n        sub.add_argument("--root", default="/content/pika-h3")\n        if action == "install":\n            sub.add_argument("--python")\n        if action == "generate":\n            prompt = sub.add_mutually_exclusive_group(required=True)\n            prompt.add_argument("--prompt")\n            prompt.add_argument("--prompt-file", type=Path)\n            sub.add_argument("--task", choices=TASKS, default="t2va")\n            sub.add_argument("--resolution", choices=("480p", "768p"), default="480p")\n            sub.add_argument("--aspect", choices=tuple(LOCK["resolution_buckets"]["480p"]), default="16:9")\n            sub.add_argument("--duration", type=int, default=5)\n            sub.add_argument("--seed", type=int, default=0)\n            sub.add_argument("--first-image")\n            sub.add_argument("--last-image")\n            sub.add_argument("--ref", action="append", default=[], dest="refs")\n            sub.add_argument("--cpu-offload", action="store_true")\n    args = parser.parse_args()\n    workspace = PikaWorkspace(args.root)\n    if args.action == "generate":\n        options = vars(args).copy()\n        for name in ("action", "root", "prompt_file"):\n            options.pop(name)\n        if args.prompt_file:\n            options["prompt"] = args.prompt_file.read_text(encoding="utf-8")\n        result = workspace.generate(**options)\n    elif args.action == "install":\n        result = workspace.install(python=args.python)\n    else:\n        result = getattr(workspace, args.action)()\n    print(json.dumps(result, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'pika_h3_lock.json': '{\n  "code_repo": "https://github.com/Pika-Labs/Pika-H3-Flex.git",\n  "code_revision": "64d81b2e98aa8aa1cf078b07a6ef3aaeae1033da",\n  "model_repo": "Pika-Labs/Pika-H3-Flex",\n  "model_revision": "0ddf55ad7067ca1e76812253bd7b4600df87f759",\n  "base_revision": "42ed227ee7df40d41602854ae760620d6eb651fe",\n  "python": "3.12",\n  "sampling_knots": [1000, 750, 500, 250, 100, 40, 0],\n  "resolution_buckets": {\n    "480p": {"21:9": [960, 416], "16:9": [832, 480], "3:2": [704, 480], "4:3": [640, 480], "5:4": [608, 480], "1:1": [480, 480], "4:5": [480, 608], "3:4": [480, 640], "2:3": [480, 704], "9:16": [480, 832], "9:21": [416, 960]},\n    "768p": {"21:9": [1536, 672], "16:9": [1344, 768], "3:2": [1152, 768], "4:3": [1024, 768], "5:4": [960, 768], "1:1": [768, 768], "4:5": [768, 960], "3:4": [768, 1024], "2:3": [768, 1152], "9:16": [768, 1344], "9:21": [672, 1536]}\n  }\n}\n'}
toolkit = Path(TOOLKIT_ROOT)
toolkit.mkdir(parents=True, exist_ok=True)
for filename, source in PIKA_SOURCES.items():
    (toolkit / filename).write_text(source)
if str(toolkit) not in sys.path:
    sys.path.insert(0, str(toolkit))
from pika_h3 import PikaWorkspace
workspace = PikaWorkspace(WORKSPACE_ROOT)
print(json.dumps(workspace.plan(), indent=2, ensure_ascii=False))


In [ ]:
# @title 2. 明確開啟安裝（不下載模型權重）
INSTALL_PIKA = False # @param {type:"boolean"}
PYTHON_312 = "" # @param {type:"string"}
if INSTALL_PIKA:
    print(json.dumps(workspace.install(python=PYTHON_312 or None), indent=2, ensure_ascii=False))
else:
    print("INSTALL_PIKA=False；未安裝。安裝日誌：<root>/install.log。")


## B. 每支影片的設定與生成

素材放到本 runtime，填實際路徑。`t2va` 不帶素材；`i2va` 填首圖；`l2va` 填尾圖；`fl2va` 填首尾兩圖。`ref2va` 填有序 JSON，例如 `["image:/content/actor.png", "video:/content/motion.mp4", "audio:/content/voice.wav"]`，至少一張圖或一段影片，與首尾圖欄位互斥。Prompt 原文傳入官方工具。

先執行設定格看實際尺寸／模型幀數與命令。每次生成建立新 run，不自動重試或換模型；官方 CLI 每次重新載入 pipeline。可從 Files 追蹤 `runs/<id>/generation.log`／`receipt.json`，執行期間配合外部 runtime watcher。


In [ ]:
# @title 3. 工作設定與離線預覽
PIKA_JOB = None
TASK = "t2va" # @param ["t2va", "i2va", "l2va", "fl2va", "ref2va"]
PROMPT = "A small red toy train rolls steadily across a wooden table. Static wide camera, soft daylight. Sound: wheels rattle softly. Music: none." # @param {type:"string"}
RESOLUTION = "480p" # @param ["480p", "768p"]
ASPECT = "16:9" # @param ["21:9", "16:9", "3:2", "4:3", "5:4", "1:1", "4:5", "3:4", "2:3", "9:16", "9:21"]
DURATION = 5 # @param {type:"integer"}
SEED = 0 # @param {type:"integer"}
FIRST_IMAGE = "" # @param {type:"string"}
LAST_IMAGE = "" # @param {type:"string"}
REFERENCES_JSON = "[]" # @param {type:"string"}
CPU_OFFLOAD = False # @param {type:"boolean"}
_pika_job = dict(prompt=PROMPT, task=TASK, resolution=RESOLUTION, aspect=ASPECT,
                 duration=DURATION, seed=SEED, first_image=FIRST_IMAGE or None,
                 last_image=LAST_IMAGE or None, refs=json.loads(REFERENCES_JSON),
                 cpu_offload=CPU_OFFLOAD)
print(json.dumps(workspace.prepare(**_pika_job), indent=2, ensure_ascii=False))
PIKA_JOB = _pika_job


In [ ]:
# @title 4. 明確開啟生成（每次執行是新的 take）
RUN_GENERATION = False # @param {type:"boolean"}
if RUN_GENERATION:
    if globals().get("PIKA_JOB") is None:
        raise RuntimeError("請先成功執行第 3 格工作設定，再開啟生成。")
    PIKA_RESULT = workspace.generate(**PIKA_JOB)
    print(json.dumps(PIKA_RESULT, indent=2, ensure_ascii=False))
else:
    print("RUN_GENERATION=False；未下載權重或生成。")


## 保存與釋放

保存整個 `<root>/runs/<id>/`：prompt、request、generation log、官方 `video.mp4`／`video.json` 與 receipt。`technically_valid` 只表示影音完整性與尺寸／幀數檢查通過；仍需播放、聽音訊與內容驗收。`/content` 會隨 runtime 刪除，請及時下載。斷線後先查原 run／PID／日誌再決定新的 take。

本工具沒有常駐 server，不管理付費 runtime。保存後另用 Colab 選單或既有管理工具停止／刪除 runtime 並核實釋放。


In [ ]:
# @title 5. 讀取已記錄狀態（不重送生成）
print(json.dumps(workspace.status(), indent=2, ensure_ascii=False))
